# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library, following the Croissant schema specification and referencing all dataset elements by their `@id` as required.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and display metadata (as attributes, not dictionary subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs as defined by their `@id` in the Croissant schema.

In [ ]:
# List all record sets in the metadata
print('Available RecordSets (by @id):')
for rs in dataset.metadata.record_sets:
    print(f"- {rs['@id']}  name: {rs['name']}")

# For each record set, show its fields and columns
for rs in dataset.metadata.record_sets:
    print(f"\nRecordSet {rs['@id']}: {rs['name']}")
    print('  Fields:')
    for field in rs['fields']:
        print(f"    - field @id: {field['@id']}  name: {field['name']}  dtype: {field.get('dataType', 'N/A')}")
        if 'columns' in field:
            for col in field['columns']:
                print(f"      - column @id: {col['@id']} name: {col['name']}  dtype: {col.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the `@id` values shown above.

In [ ]:
# Choose record sets to extract by their @id (use output above; fill as appropriate)
# If you are not sure, try with the first available record set
record_sets_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
print('Available record sets:', record_sets_ids)

dataframes = {}
for rsid in record_sets_ids:
    print(f'Loading records for RecordSet @id: {rsid}')
    records_iter = dataset.records(record_set=rsid)
    records = list(records_iter)
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f'\nFields for {rsid}:', df.columns.tolist())
        display(df.head())
    else:
        print(f'No records found for {rsid}')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping.

In [ ]:
# For demonstration, choose the first DataFrame (if any)

if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]
    print(f'Selected record set: {main_record_set_id}')
    print('Available fields:', df.columns.tolist())
    
    # Automatically select a numeric column to analyze
    numeric_cols = df.select_dtypes(include=[int, float]).columns.tolist()
    if not numeric_cols:
        print('No numeric columns found for EDA.')
    else:
        numeric_field_id = numeric_cols[0]  # You can replace with another numeric field @id as required
        print(f'Analyzing numeric field: {numeric_field_id}')
        
        # Example filtering - top 10 values
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() < df[numeric_field_id].max() else df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:\n", filtered_df.head())
        
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Choose group field as the first non-numeric column
        non_numeric_cols = [col for col in df.columns if col not in numeric_cols]
        if non_numeric_cols:
            group_field = non_numeric_cols[0]
            try:
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"\nGrouped data by '{group_field}':")
                print(grouped_df.head())
            except Exception as e:
                print(f'Could not group by {group_field}: {e}')
else:
    print('No dataframes to analyze.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = list(dataframes.values())[0]
    if not df.empty:
        # Plot histogram for the first numeric column
        numeric_cols = df.select_dtypes(include=[int, float]).columns.tolist()
        if numeric_cols:
            field = numeric_cols[0]
            plt.figure(figsize=(8, 4))
            sns.histplot(df[field].dropna(), kde=True)
            plt.title(f'Distribution of numeric field: {field}')
            plt.xlabel(field)
            plt.ylabel('Frequency')
            plt.show()

        # If at least two numeric fields, scatterplot
        if len(numeric_cols) >= 2:
            plt.figure(figsize=(8, 6))
            sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
            plt.xlabel(numeric_cols[0])
            plt.ylabel(numeric_cols[1])
            plt.title(f'Scatterplot of {numeric_cols[0]} vs {numeric_cols[1]}')
            plt.show()
    else:
        print('Selected DataFrame is empty. Cannot plot.')
else:
    print('No dataframes to visualize.')

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and analyze the FAIR² mass spectrometry dataset using the `mlcroissant` library, referencing all entities by their `@id`. You can adapt this notebook for your specific record set and field selection, applying further domain-specific analysis as needed.